In [ ]:
!pip install -q pymupdf sentence-transformers pinecone-client langchain

In [ ]:
pip install --upgrade pinecone

In [ ]:
import os

# Path where the ZIP file was extracted
extract_path = "/content/Policies"

# Recursively find all PDF files (handles subfolders as well)
pdf_files = []
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(".pdf"):
            pdf_files.append(os.path.join(root, file))

# Display the total number of PDF files
print(f"✅ Total number of PDF documents extracted: {len(pdf_files)}\n")

# Display the list of files
print("📄 List of Extracted PDF Files:")
for i, file in enumerate(pdf_files, 1):
    print(f"{i}. {os.path.basename(file)}")

✅ Total number of PDF documents extracted: 31

📄 List of Extracted PDF Files:
1. Issues with Water Purifiers - Amazon Customer Service.pdf
2. Pay by Invoice Policies - Amazon Customer Service.pdf
3. Tamper-evident packaging_ Identify if your package was tampered - Amazon Customer Service.pdf
4. jobs.amazon.in_privacy-notice.pdf
5. amazon-vendor-enrollment-and-contact-sop-2024.pdf
6. Free shipping on subsequent orders placed within 4-hours - Amazon Customer Service.pdf
7. Conditions of Use - Amazon Customer Service.pdf
8. Check your eligibility for Guaranteed Delivery - Amazon Customer Service.pdf
9. amazon-business-faq.pdf
10. About Amazon Fast Shipping Options_ Same-Day & One-Day Delivery - Amazon Customer Service.pdf
11. Amazon Delivery Charges and Shipping Speeds - Amazon Customer Service.pdf
12. Cancel Items and Orders - Amazon Customer Service.pdf
13. Issues with Camera FAQs - Amazon Customer Service.pdf
14. Amazon Refund Policy_ Payment Methods & Processing Times - Amazon Custome

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Document ID": range(1, len(pdf_files) + 1),
    "File Name": [os.path.basename(f) for f in pdf_files],
    "Full Path": pdf_files
})

df

,Document ID,File Name,Full Path
0,1,Issues with Water Purifiers - Amazon Customer ...,/content/Policies/POLICIES/Issues with Water P...
1,2,Pay by Invoice Policies - Amazon Customer Serv...,/content/Policies/POLICIES/Pay by Invoice Poli...
2,3,Tamper-evident packaging_ Identify if your pac...,/content/Policies/POLICIES/Tamper-evident pack...
3,4,jobs.amazon.in_privacy-notice.pdf,/content/Policies/POLICIES/jobs.amazon.in_priv...
4,5,amazon-vendor-enrollment-and-contact-sop-2024.pdf,/content/Policies/POLICIES/amazon-vendor-enrol...
5,6,Free shipping on subsequent orders placed with...,/content/Policies/POLICIES/Free shipping on su...
6,7,Conditions of Use - Amazon Customer Service.pdf,/content/Policies/POLICIES/Conditions of Use -...
7,8,Check your eligibility for Guaranteed Delivery...,/content/Policies/POLICIES/Check your eligibil...
8,9,amazon-business-faq.pdf,/content/Policies/POLICIES/amazon-business-faq...
9,10,About Amazon Fast Shipping Options_ Same-Day &...,/content/Policies/POLICIES/About Amazon Fast S...


# Step 3: Extract Text from Each PDF with Metadata (source and page number)

In [ ]:
!pip install -q pymupdf

In [ ]:
import os
import fitz  # PyMuPDF
from tqdm import tqdm

In [ ]:
# Path where the ZIP file was extracted
extract_path = "/content/Policies"

# Collect all PDF file paths (including subfolders)
pdf_files = []
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(".pdf"):
            pdf_files.append(os.path.join(root, file))

print(f"Total PDF files to process: {len(pdf_files)}")

Total PDF files to process: 31


In [ ]:
def extract_text_from_pdfs(pdf_paths):
    documents = []

    for pdf_path in tqdm(pdf_paths, desc="Processing PDFs"):
        file_name = os.path.basename(pdf_path)

        try:
            doc = fitz.open(pdf_path)

            for page_num in range(len(doc)):
                page = doc[page_num]
                text = page.get_text("text")

                # Clean and validate text
                if text and text.strip():
                    documents.append({
                        "text": text.strip(),
                        "metadata": {
                            "source": file_name,
                            "page": page_num + 1,
                            "file_path": pdf_path
                        }
                    })

            doc.close()

        except Exception as e:
            print(f"❌ Error processing {file_name}: {e}")

    return documents

In [ ]:
documents = extract_text_from_pdfs(pdf_files)

print(f"\n✅ Total pages extracted: {len(documents)}")

Processing PDFs: 100%|██████████| 31/31 [00:00<00:00, 43.49it/s]


✅ Total pages extracted: 147


In [ ]:
# Display a few sample entries to verify extraction
for i, doc in enumerate(documents[:3]):
    print(f"\n📄 Document {i+1}")
    print(f"Source : {doc['metadata']['source']}")
    print(f"Page   : {doc['metadata']['page']}")
    print(f"Text Preview:\n{doc['text'][:300]}")
    print("-" * 80)


📄 Document 1
Source : Issues with Water Purifiers - Amazon Customer Service.pdf
Page   : 1
Text Preview:
Issues with Water Purifiers
Find quick resolution and troubleshooting steps for Water Purifier.
To track status of your package, go to
I am unable to turn ON the purifier.
Potential root cause: Faulty power source.
Remedy: Check the power outlet.
1. Connect the unit into a different wall outlet.
2. 
--------------------------------------------------------------------------------

📄 Document 2
Source : Issues with Water Purifiers - Amazon Customer Service.pdf
Page   : 2
Text Preview:
Potential root cause: Fault in check valve.
Remedy: Contact the manufacturer.
1. Contact the manufacturer for further assistance.
I am unable to stop my water filter from leaking.
Potential root cause: Membrane is leaking water around the connection
points.
Remedy: Check the following procedure.
1. 
--------------------------------------------------------------------------------

📄 Document 3
Source : I

In [ ]:
from collections import Counter

page_count = Counter([doc['metadata']['source'] for doc in documents])

print("\n📊 Pages Extracted per Document:")
for source, count in page_count.items():
    print(f"{source}: {count} pages")


📊 Pages Extracted per Document:
Issues with Water Purifiers - Amazon Customer Service.pdf: 3 pages
Pay by Invoice Policies - Amazon Customer Service.pdf: 2 pages
Tamper-evident packaging_ Identify if your package was tampered - Amazon Customer Service.pdf: 2 pages
jobs.amazon.in_privacy-notice.pdf: 4 pages
amazon-vendor-enrollment-and-contact-sop-2024.pdf: 8 pages
Free shipping on subsequent orders placed within 4-hours - Amazon Customer Service.pdf: 2 pages
Conditions of Use - Amazon Customer Service.pdf: 17 pages
Check your eligibility for Guaranteed Delivery - Amazon Customer Service.pdf: 1 pages
amazon-business-faq.pdf: 4 pages
About Amazon Fast Shipping Options_ Same-Day & One-Day Delivery - Amazon Customer Service.pdf: 3 pages
Amazon Delivery Charges and Shipping Speeds - Amazon Customer Service.pdf: 2 pages
Cancel Items and Orders - Amazon Customer Service.pdf: 2 pages
Issues with Camera FAQs - Amazon Customer Service.pdf: 8 pages
Amazon Refund Policy_ Payment Methods & Process

In [ ]:
import json

output_path = "/content/extracted_documents.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"\n💾 Extracted documents saved to: {output_path}")


💾 Extracted documents saved to: /content/extracted_documents.json


# Step 4: Text Cleaning

In [ ]:
import re
from tqdm import tqdm

In [ ]:
def clean_text(text):
    """
    Cleans extracted PDF text to improve embedding quality.
    """
    if not text:
        return ""

    # 1. Remove hyphenation at line breaks (e.g., "deliv-\nery" -> "delivery")
    text = re.sub(r'-\s*\n\s*', '', text)

    # 2. Replace newlines and tabs with spaces
    text = re.sub(r'[\n\r\t]+', ' ', text)

    # 3. Remove multiple spaces
    text = re.sub(r'\s{2,}', ' ', text)

    # 4. Remove page numbers or lines with only digits
    text = re.sub(r'\b\d+\b', '', text)

    # 5. Remove common boilerplate phrases (customize as needed)
    boilerplate_patterns = [
        r'©\s*\d{4}.*?Amazon.*?',
        r'All rights reserved\.?',
        r'Page\s*\d+\s*of\s*\d+',
    ]
    for pattern in boilerplate_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # 6. Normalize special characters to ASCII
    text = text.encode('ascii', 'ignore').decode('ascii')

    # 7. Final whitespace cleanup
    text = text.strip()

    return text

In [ ]:
cleaned_documents = []

for doc in tqdm(documents, desc="Cleaning text"):
    cleaned_text = clean_text(doc["text"])

    # Keep only meaningful text
    if cleaned_text:
        cleaned_documents.append({
            "text": cleaned_text,
            "metadata": doc["metadata"]
        })

print(f"✅ Total cleaned documents: {len(cleaned_documents)}")

Cleaning text: 100%|██████████| 147/147 [00:00<00:00, 4175.08it/s]

✅ Total cleaned documents: 147


In [ ]:
# Preview comparison for the first document
print("🔹 BEFORE CLEANING:\n")
print(documents[0]["text"][:500])

print("\n" + "="*80 + "\n")

print("🔹 AFTER CLEANING:\n")
print(cleaned_documents[0]["text"][:500])

🔹 BEFORE CLEANING:

Issues with Water Purifiers
Find quick resolution and troubleshooting steps for Water Purifier.
To track status of your package, go to
I am unable to turn ON the purifier.
Potential root cause: Faulty power source.
Remedy: Check the power outlet.
1. Connect the unit into a different wall outlet.
2. Ensure the power button is ON.
3. Use a tester to check whether there is proper supply of electric current to the point.
4. Make sure the plug has been inserted properly.
Potential root cause: Physica


🔹 AFTER CLEANING:

Issues with Water Purifiers Find quick resolution and troubleshooting steps for Water Purifier. To track status of your package, go to I am unable to turn ON the purifier. Potential root cause: Faulty power source. Remedy: Check the power outlet. . Connect the unit into a different wall outlet. . Ensure the power button is ON. . Use a tester to check whether there is proper supply of electric current to the point. . Make sure the plug has been inserted p

In [ ]:
import json

output_path = "/content/cleaned_documents.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(cleaned_documents, f, ensure_ascii=False, indent=2)

print(f"💾 Cleaned documents saved to: {output_path}")

💾 Cleaned documents saved to: /content/cleaned_documents.json


# Step 5: Text Chunking

Large documents like Amazon policy PDFs cannot be embedded as a whole due to:

Token limits of embedding and LLM models.
Reduced retrieval precision when chunks are too large.
Loss of context when chunks are too small.

Effective chunking ensures that each piece of text is semantically meaningful, searchable, and contextually coherent.

In [ ]:
!pip install -q langchain_text_splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

In [ ]:
# Define chunking parameters
chunk_size = 1000        # Maximum characters per chunk
chunk_overlap = 150      # Overlap between consecutive chunks

# Initialize the recursive text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", ".", " ", ""]
)

In [ ]:
chunked_documents = []

for doc in cleaned_documents:
    chunks = text_splitter.split_text(doc["text"])

    for i, chunk in enumerate(chunks):
        chunked_documents.append({
            "text": chunk,
            "metadata": {
                "source": doc["metadata"]["source"],
                "page": doc["metadata"]["page"],
                "chunk_id": i
            }
        })

print(f"✅ Total number of chunks created: {len(chunked_documents)}")

✅ Total number of chunks created: 382


In [ ]:
# Display the first 3 chunks
for i, chunk in enumerate(chunked_documents[:3]):
    print(f"\n📌 Chunk {i+1}")
    print(f"Source : {chunk['metadata']['source']}")
    print(f"Page   : {chunk['metadata']['page']}")
    print(f"Chunk ID: {chunk['metadata']['chunk_id']}")
    print(f"Text Preview:\n{chunk['text'][:500]}")
    print("-" * 80)


📌 Chunk 1
Source : Issues with Water Purifiers - Amazon Customer Service.pdf
Page   : 1
Chunk ID: 0
Text Preview:
Issues with Water Purifiers Find quick resolution and troubleshooting steps for Water Purifier. To track status of your package, go to I am unable to turn ON the purifier. Potential root cause: Faulty power source. Remedy: Check the power outlet. . Connect the unit into a different wall outlet. . Ensure the power button is ON. . Use a tester to check whether there is proper supply of electric current to the point. . Make sure the plug has been inserted properly. Potential root cause: Physical da
--------------------------------------------------------------------------------

📌 Chunk 2
Source : Issues with Water Purifiers - Amazon Customer Service.pdf
Page   : 1
Chunk ID: 1
Text Preview:
. Check the RO storage tank to make sure it's properly pressurized. Note: Pressure that's too high or too low could affect the water flow. b. Always check the manual/user guide for the spe

In [ ]:
import json

output_path = "/content/chunked_documents.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(chunked_documents, f, ensure_ascii=False, indent=2)

print(f"💾 Chunked documents saved to: {output_path}")

💾 Chunked documents saved to: /content/chunked_documents.json


# Step 5: Generating Embeddings

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import numpy as np

In [ ]:
# Load the embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Verify embedding dimension
embedding_dimension = embedding_model.get_sentence_embedding_dimension()
print(f"✅ Embedding dimension: {embedding_dimension}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding dimension: 384


In [ ]:
texts = [doc["text"] for doc in chunked_documents]
print(f"Total chunks to embed: {len(texts)}")

Total chunks to embed: 382


In [ ]:
# Generate embeddings with progress bar
embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # Important for cosine similarity
)

print(f"✅ Embeddings generated with shape: {embeddings.shape}")

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Embeddings generated with shape: (382, 384)


In [ ]:
embeddings

array([[-0.04517289, -0.06796417, -0.03327918, ...,  0.02851642,
        -0.0236332 ,  0.02142608],
       [ 0.02960823, -0.09815159, -0.05684664, ...,  0.03684441,
        -0.1332547 ,  0.00980028],
       [-0.03185693, -0.01679057, -0.01434712, ..., -0.03722099,
         0.0522316 , -0.01412621],
       ...,
       [-0.07097777,  0.07506189,  0.04634284, ..., -0.01737394,
         0.06797408, -0.07509985],
       [-0.08446255,  0.04307471,  0.03866115, ..., -0.0148195 ,
         0.03539835, -0.03224996],
       [-0.07249718,  0.06968575,  0.00083227, ..., -0.01188172,
         0.04643731, -0.09470257]], dtype=float32)

In [ ]:
vectors = []

for i, (doc, embedding) in enumerate(zip(chunked_documents, embeddings)):
    vector_id = f"{doc['metadata']['source']}_p{doc['metadata']['page']}_c{i}"

    vectors.append({
        "id": vector_id,
        "values": embedding.tolist(),
        "metadata": {
            "text": doc["text"],
            "source": doc["metadata"]["source"],
            "page": doc["metadata"]["page"],
            "chunk_id": doc["metadata"]["chunk_id"]
        }
    })

print(f"✅ Prepared {len(vectors)} vectors for Pinecone.")

✅ Prepared 382 vectors for Pinecone.


In [ ]:
# Display a sample vector
sample_vector = vectors[0]

print("Sample Vector ID:", sample_vector["id"])
print("Metadata:", sample_vector["metadata"])
print("First 10 Embedding Values:", sample_vector["values"][:10])
print("Embedding Length:", len(sample_vector["values"]))

Sample Vector ID: Issues with Water Purifiers - Amazon Customer Service.pdf_p1_c0
Metadata: {'text': "Issues with Water Purifiers Find quick resolution and troubleshooting steps for Water Purifier. To track status of your package, go to I am unable to turn ON the purifier. Potential root cause: Faulty power source. Remedy: Check the power outlet. . Connect the unit into a different wall outlet. . Ensure the power button is ON. . Use a tester to check whether there is proper supply of electric current to the point. . Make sure the plug has been inserted properly. Potential root cause: Physical damage. Remedy: Check for the physical damage. . Check and ensure that there is no physical damage to the product. . Make sure there is no twist or break to the power cable. I am unable to acquire water from my purifier. Potential root cause: Unknown. Remedy: Check the following procedure. . Check the RO Storage Tank Pressure: a. Check the RO storage tank to make sure it's properly pressurized. No

In [ ]:
import json

output_path = "/content/embeddings.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(vectors, f)

print(f"💾 Embeddings saved to: {output_path}")

💾 Embeddings saved to: /content/embeddings.json


# Step 6: Upload Embeddings to Pinecone

In [ ]:
!pip install -q pinecone

In [ ]:
from pinecone import Pinecone, ServerlessSpec
import os

In [ ]:
import os
from getpass import getpass

os.environ["PINECONE_API_KEY"] = getpass("Enter your Pinecone API Key: ")

Enter your Pinecone API Key: ··········


In [ ]:
# Initialize Pinecone client
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

In [ ]:
index_name = "amazon-policies"
embedding_dimension = 384  # For all-MiniLM-L6-v2

# Create the index if it doesn't already exist
existing_indexes = [index.name for index in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=embedding_dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",       # Options: "aws", "gcp", "azure"
            region="us-east-1" # Choose a region supported by your Pinecone account
        )
    )
    print(f"✅ Index '{index_name}' created successfully.")
else:
    print(f"ℹ️ Index '{index_name}' already exists.")

✅ Index 'amazon-policies' created successfully.


In [ ]:
# Connect to the index
index = pc.Index(index_name)
print("✅ Connected to Pinecone index.")

✅ Connected to Pinecone index.


In [ ]:
vectors_to_upsert = [
    (vec["id"], vec["values"], vec["metadata"])
    for vec in vectors
]

In [ ]:
from tqdm import tqdm

batch_size = 100

for i in tqdm(range(0, len(vectors_to_upsert), batch_size), desc="Uploading to Pinecone"):
    batch = vectors_to_upsert[i:i + batch_size]
    index.upsert(vectors=batch)

print("✅ All vectors uploaded successfully!")

Uploading to Pinecone: 100%|██████████| 4/4 [00:02<00:00,  1.44it/s]

✅ All vectors uploaded successfully!


In [ ]:
# Get index statistics
stats = index.describe_index_stats()
print("📊 Index Statistics:")
print(stats)

📊 Index Statistics:
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '185',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 09 Apr 2026 20:28:36 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '41',
                                    'x-pinecone-request-latency-ms': '41',
                                    'x-pinecone-response-duration-ms': '42'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 382}},
 'storageFullness': 0.0,
 'total_vector_count': 382,
 'vector_type': 'dense'}


In [ ]:
# Example query
query = "What is the return period for items?"

# Generate embedding for the query
query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
).tolist()

# Query Pinecone
results = index.query(
    vector=query_embedding,
    top_k=5,
    include_metadata=True
)

# Display results
for i, match in enumerate(results["matches"], 1):
    print(f"\nResult {i}")
    print(f"Score  : {match['score']:.4f}")
    print(f"Source : {match['metadata']['source']}")
    print(f"Page   : {match['metadata']['page']}")
    print(f"Text   : {match['metadata']['text'][:300]}")


Result 1
Score  : 0.6018
Source : Amazon.in Returns Policy - Amazon Customer Service.pdf
Page   : 1
Text   : Amazon.in Returns Policy Information on return eligibility, timelines and other terms & conditions for items purchased on Amazon.in. Disclaimer: In the event of any discrepancy or conflict, the English version will prevail over the translation. Most items purchased from sellers listed on Amazon.in a

Result 2
Score  : 0.5889
Source : Conditions of Use - Amazon Customer Service.pdf
Page   : 15
Text   : .in are returnable within the return window, except those that are explicitly identified as not returnable. The return is processed only if: it is determined that the product was not damaged while in your possession; the product is not different from what was shipped to you; the product is returned 

Result 3
Score  : 0.5593
Source : Amazon.in Returns Policy - Amazon Customer Service.pdf
Page   : 4
Text   : . You can also return the item within /  days of delivery for full refund.